##### Copyright 2020 The TensorFlow Authors.

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# TensorFlow Recommenders: Quickstart

<table class="tfo-notebook-buttons" align="left">
  <td>
    <a target="_blank" href="https://www.tensorflow.org/recommenders/quickstart"><img src="https://www.tensorflow.org/images/tf_logo_32px.png" />View on TensorFlow.org</a>
  </td>
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/tensorflow/recommenders/blob/main/docs/examples/quickstart.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/tensorflow/recommenders/blob/main/docs/examples/quickstart.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
  </td>
  <td>
    <a href="https://storage.googleapis.com/tensorflow_docs/recommenders/docs/examples/quickstart.ipynb"><img src="https://www.tensorflow.org/images/download_logo_32px.png" />Download notebook</a>
  </td>
</table>

In this tutorial, we build a simple matrix factorization model using the [MovieLens 100K dataset](https://grouplens.org/datasets/movielens/100k/) with TFRS. We can use this model to recommend movies for a given user.

### Import TFRS

First, install and import TFRS:

In [ ]:
!pip install -q tensorflow-recommenders
!pip install -q --upgrade tensorflow-datasets

In [ ]:
from typing import Dict, Text

import numpy as np
import tensorflow as tf

import tensorflow_datasets as tfds
import tensorflow_recommenders as tfrs

### Read the data

In [ ]:
# Ratings data.
ratings = tfds.load('movielens/100k-ratings', split="train")
# Features of all the available movies.
movies = tfds.load('movielens/100k-movies', split="train")

# Select the basic features.
ratings = ratings.map(lambda x: {
    "movie_title": x["movie_title"],
    "user_id": x["user_id"]
})
movies = movies.map(lambda x: x["movie_title"])

In [ ]:
len(ratings)
len(movies)

In [ ]:
for r in ratings.take(10):  # Change 5 to however many you want
    print(r)

In [ ]:
for r in movies.take(5):  # Change 5 to however many you want tp print
    print(r)
    print(type(r))

Build vocabularies to convert user ids and movie titles into integer indices for embedding layers:

In [ ]:
# creating vocab for user and movie embedding matrix separately. string lookup converts the strings into the integer indices or numpy indices. It only adds the index not necessarily convert to integer

user_ids_vocabulary = tf.keras.layers.StringLookup(mask_token=None)
user_ids_vocabulary.adapt(ratings.map(lambda x: x["user_id"]))

movie_titles_vocabulary = tf.keras.layers.StringLookup(mask_token=None)
movie_titles_vocabulary.adapt(movies)

In [ ]:
# why there are only 6 user id's

len(user_ids_vocabulary.get_vocabulary())

In [ ]:
print(user_ids_vocabulary.get_vocabulary()[:5])

In [ ]:
len(movie_titles_vocabulary.get_vocabulary())

In [ ]:
print(movie_titles_vocabulary.get_vocabulary()[:5])

### Define a model

We can define a TFRS model by inheriting from `tfrs.Model` and implementing the `compute_loss` method:

In [ ]:
class MovieLensModel(tfrs.Model):
  # We derive from a custom base class to help reduce boilerplate. Under the hood,
  # these are still plain Keras Models.

  def __init__(
      self,
      user_model: tf.keras.Model,
      movie_model: tf.keras.Model,
      task: tfrs.tasks.Retrieval):
    super().__init__()

    # Set up user and movie representations.
    self.user_model = user_model
    self.movie_model = movie_model

    # Set up a retrieval task.
    self.task = task

  def compute_loss(self, features: Dict[Text, tf.Tensor], training=False) -> tf.Tensor:
    # Define how the loss is computed.

    user_embeddings = self.user_model(features["user_id"])
    movie_embeddings = self.movie_model(features["movie_title"])

    return self.task(user_embeddings, movie_embeddings)

Define the two models and the retrieval task.

In [ ]:
# Define user embedding model, by passing unique user id vocab and embedding size of of 64, so embedding size is 944*64

user_model = tf.keras.Sequential([
    user_ids_vocabulary,
    tf.keras.layers.Embedding(len(user_ids_vocabulary.get_vocabulary()), 64)
])

In [ ]:
print(user_model.layers)

In [ ]:
# define movies embedding model, by passing movie vocab and 64 as embedding size so embedding size is 1665 * 64

movie_model = tf.keras.Sequential([
    movie_titles_vocabulary,
    tf.keras.layers.Embedding(len(movie_titles_vocabulary.get_vocabulary()), 64)
])

In [ ]:
print(movie_model.layers)

In [ ]:
# Access the Embedding layer inside the Sequential model
embedding_layer = movie_model.layers[1]

# Get the weights (embeddings) from the Embedding layer
embedding_weights = embedding_layer.get_weights()[0]

# Print the full embedding matrix (can be large!)
print(embedding_weights)

# Or, print the shape of the matrix
print("Embedding shape:", embedding_weights.shape)


In [ ]:
movies

In [ ]:
# Define your objectives - retrieval task,, what i think is happening here is, the embedding matrix is1665*64, i think embedding size is 64 and vocab is 1665. But the
# task = tfrs.tasks.Retrieval(metrics=tfrs.metrics.FactorizedTopK(
#     movies.batch(128).map(movie_model)
#   )
# )


### Fit and evaluate it.

Create the model, train it, and generate predictions:



In [ ]:
# # Create a retrieval model.
# model = MovieLensModel(user_model, movie_model, task)
# model.compile(optimizer=tf.keras.optimizers.Adagrad(0.5))

# # Train for 3 epochs.
# model.fit(ratings.batch(4096), epochs=3)

# # Use brute-force search to set up retrieval using the trained representations.
# index = tfrs.layers.factorized_top_k.BruteForce(model.user_model)
# index.index_from_dataset(
#     movies.batch(100).map(lambda title: (title, model.movie_model(title))))

# # Get some recommendations.
# _, titles = index(np.array(["42"]))
# print(f"Top 3 recommendations for user 42: {titles[0, :3]}")